# EA4 — Transición Energética y Vehículos Eléctricos

**Autor:** Fredy Ramírez  
**Pregunta guía:** ¿De qué manera influyen la temperatura ambiental y la hora del día en la eficiencia de recarga y la autonomía proyectada de las flotas de vehículos eléctricos?

Este cuaderno documenta la ingesta, auditoría, limpieza, análisis y verificación de datos. Los hallazgos se redactarán después de ejecutar el análisis; no se inventarán resultados.

## 1. Configuración e importaciones

Ejecuta las celdas en orden. En el terminal, crea y activa un entorno virtual e instala las dependencias desde `requirements.txt`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os

DATA_DIR = Path('datos')
DATA_DIR.mkdir(exist_ok=True)
DATASET_URL = 'https://huggingface.co/datasets/ProWiddan/Global-EV-Charging-Station-Telemetry'
print('Origen del dataset:', DATASET_URL)

## 2. Ingesta de datos desde Hugging Face

Esta celda intenta localizar archivos de datos CSV/JSON/Parquet del repositorio del dataset. Si el repositorio requiere una configuración distinta, revisaremos el listado de archivos y adaptaremos el cargador.

In [ ]:
# Instala huggingface_hub si aún no está disponible: pip install huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download

dataset_repo = 'ProWiddan/Global-EV-Charging-Station-Telemetry'
files = list_repo_files(dataset_repo, repo_type='dataset')
data_files = [f for f in files if f.lower().endswith(('.csv', '.json', '.parquet'))]
print('Archivos de datos encontrados:')
for f in data_files:
    print('-', f)

if not data_files:
    raise FileNotFoundError('No se encontraron CSV, JSON o Parquet en el repositorio. Revisa el listado de archivos del dataset.')

chosen_file = data_files[0]
local_file = hf_hub_download(repo_id=dataset_repo, filename=chosen_file, repo_type='dataset', local_dir=DATA_DIR)
print('Archivo descargado:', local_file)

In [ ]:
# Carga según extensión
ext = Path(chosen_file).suffix.lower()
if ext == '.csv':
    df = pd.read_csv(local_file)
elif ext == '.json':
    df = pd.read_json(local_file)
elif ext == '.parquet':
    df = pd.read_parquet(local_file)
else:
    raise ValueError(f'Formato no admitido: {ext}')

display(df.head())
print('Dimensiones (filas, columnas):', df.shape)

## 3. Auditoría inicial de datos

Antes de limpiar, examinamos nombres de columnas, tipos, faltantes y duplicados. No reemplazaremos valores faltantes por cero salvo que el significado del dato justifique esa decisión.

In [ ]:
display(pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isnull().sum(), 'nulos_%': (df.isnull().mean()*100).round(2)}))
print('Filas duplicadas:', df.duplicated().sum())
print('Columnas disponibles:', df.columns.tolist())

## 4. Próximas etapas

- Identificar campos de temperatura, fecha/hora, energía, duración, potencia, vehículo y batería.
- Normalizar unidades y documentar datos faltantes.
- Crear variables derivadas con Pandas y NumPy (solo si existen los campos necesarios).
- Segmentar con `.query()`, agrupar con `groupby()` y verificar cálculos mediante máscaras NumPy.
- Construir visualizaciones y dashboard de Streamlit.
- Redactar hallazgos numéricos y decisiones operativas basados en resultados reales.
- Exportar el resumen de auditoría a TXT.